In [12]:
import os
import torch
import torch.nn as nn
from torch.autograd import Variable
import torch.utils.data as Data
import torchvision
import torch.nn.functional as F
import numpy as np

# 超参数
learning_rate = 1e-4
keep_prob_rate = 0.7
max_epoch = 3
BATCH_SIZE = 50

print("正在加载 MNIST 数据集...")

正在加载 MNIST 数据集...


In [13]:
# 下载 MNIST 数据
train_data = torchvision.datasets.MNIST(
    root='./mnist/',
    train=True,
    transform=torchvision.transforms.ToTensor(),
    download=True,
)

train_loader = Data.DataLoader(dataset=train_data, batch_size=BATCH_SIZE, shuffle=True)

test_data = torchvision.datasets.MNIST(
    root='./mnist/',
    train=False,
    transform=torchvision.transforms.ToTensor(),
    download=True,
)

# 使用前500个测试样本
test_x_list = []
test_y_list = []
for i in range(500):
    x, y = test_data[i]
    test_x_list.append(x)
    test_y_list.append(y)

test_x = torch.stack(test_x_list)
test_y = torch.stack([torch.tensor(y) for y in test_y_list])

print(f"训练集大小: {len(train_data)}")
print(f"测试集大小: {len(test_data)}")

训练集大小: 60000
测试集大小: 10000


In [14]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        # 卷积层 1
        self.conv1 = nn.Sequential(
            nn.Conv2d(
                in_channels=1,      # 输入通道数（灰度图像为1）
                out_channels=32,    # 输出通道数（32个滤波器）
                kernel_size=7,      # 卷积核大小 7x7
                stride=1,           # 步长为1
                padding=3,          # padding方式（kernel_size//2，保持尺寸不变）
            ),
            nn.ReLU(),             # 激活函数
            nn.MaxPool2d(2),       # 最大池化，2x2窗口
        )

        # 卷积层 2
        self.conv2 = nn.Sequential(
            nn.Conv2d(
                in_channels=32,     # 输入通道数（上一层输出32）
                out_channels=64,    # 输出通道数（64个滤波器）
                kernel_size=5,      # 卷积核大小 5x5
                stride=1,           # 步长为1
                padding=2,          # padding方式（kernel_size//2，保持尺寸不变）
            ),
            nn.ReLU(),             # 激活函数
            nn.MaxPool2d(2),       # 最大池化，2x2窗口
        )

        # 全连接层 1
        self.out1 = nn.Linear(7 * 7 * 64, 1024, bias=True)

        # Dropout 层
        self.dropout = nn.Dropout(keep_prob_rate)

        # 全连接层 2（输出层）
        self.out2 = nn.Linear(1024, 10, bias=True)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = x.view(x.size(0), -1)  # 展平：(batch_size, 7*7*64)
        out1 = self.out1(x)
        out1 = F.relu(out1)
        out1 = self.dropout(out1)
        out2 = self.out2(out1)
        output = F.softmax(out2, dim=1)  # 指定维度
        return output

# 创建模型
cnn = CNN()
print("模型结构:")
print(cnn)

模型结构:
CNN(
  (conv1): Sequential(
    (0): Conv2d(1, 32, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (out1): Linear(in_features=3136, out_features=1024, bias=True)
  (dropout): Dropout(p=0.7, inplace=False)
  (out2): Linear(in_features=1024, out_features=10, bias=True)
)


In [15]:
def test(cnn):
    global prediction
    cnn.eval()  # 设置为评估模式
    with torch.no_grad():  # 不需要梯度计算
        y_pre = cnn(test_x)
        _, pre_index = torch.max(y_pre, 1)
        pre_index = pre_index.view(-1)
        prediction = pre_index.data.cpu().numpy()
        correct = np.sum(prediction == test_y.cpu().numpy())
    cnn.train()  # 恢复训练模式
    return correct / 500.0

In [16]:
def train(cnn):
    optimizer = torch.optim.Adam(cnn.parameters(), lr=learning_rate)
    loss_func = nn.CrossEntropyLoss()

    print("开始训练...")
    print("=" * 60)

    for epoch in range(max_epoch):
        for step, (x_, y_) in enumerate(train_loader):
            output = cnn(x_)
            loss = loss_func(output, y_)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            if step != 0 and step % 20 == 0:
                accuracy = test(cnn)
                print(f"Epoch {epoch}, Step {step}: 测试准确率 = {accuracy:.4f}")

    print()
    print("=" * 60)
    print("训练完成！")

    # 最终评估
    final_accuracy = test(cnn)
    print(f"最终测试集准确率: {final_accuracy:.4f}")

    if final_accuracy >= 0.96:
        print("[SUCCESS] 达到目标精度 96% 以上！")
    else:
        print("[WARNING] 未达到目标精度 96%，可能需要增加训练轮数")

    print("=" * 60)

In [ ]:
# 训练模型
train(cnn)

开始训练...
Epoch 0, Step 20: 测试准确率 = 0.2600
Epoch 0, Step 40: 测试准确率 = 0.4420
Epoch 0, Step 60: 测试准确率 = 0.4940
Epoch 0, Step 80: 测试准确率 = 0.6140
Epoch 0, Step 100: 测试准确率 = 0.6600
Epoch 0, Step 120: 测试准确率 = 0.6980
Epoch 0, Step 140: 测试准确率 = 0.7340
Epoch 0, Step 160: 测试准确率 = 0.7280
Epoch 0, Step 180: 测试准确率 = 0.8020
Epoch 0, Step 200: 测试准确率 = 0.8380
Epoch 0, Step 220: 测试准确率 = 0.8420
Epoch 0, Step 240: 测试准确率 = 0.8780
Epoch 0, Step 260: 测试准确率 = 0.8900
Epoch 0, Step 280: 测试准确率 = 0.8900
Epoch 0, Step 300: 测试准确率 = 0.9000
Epoch 0, Step 320: 测试准确率 = 0.8960
Epoch 0, Step 340: 测试准确率 = 0.8980
Epoch 0, Step 360: 测试准确率 = 0.9120
Epoch 0, Step 380: 测试准确率 = 0.9100
Epoch 0, Step 400: 测试准确率 = 0.9180
Epoch 0, Step 420: 测试准确率 = 0.9140
Epoch 0, Step 440: 测试准确率 = 0.9260
Epoch 0, Step 460: 测试准确率 = 0.9260
Epoch 0, Step 480: 测试准确率 = 0.9320
Epoch 0, Step 500: 测试准确率 = 0.9380
Epoch 0, Step 520: 测试准确率 = 0.9360
Epoch 0, Step 540: 测试准确率 = 0.9300
Epoch 0, Step 560: 测试准确率 = 0.9320
Epoch 0, Step 580: 测试准确率 = 0.9420
Epoch 0, S